# ⚖️ 台灣司法官律師國考 評分與申論推導模型微調 (Fine-Tuning Law Exam Evaluator)
支援在 **Google Colab (免費 T4 GPU 或 CPU)** 上極速微調開源模型（如 Qwen2.5-7B 或 Taiwan-LLM）。
訓練資料直接從您的 Google Drive (`/content/drive/MyDrive/LawExamData/training_data`) 讀取 **100% 真實核實裁判、憲法法庭判決與法理規準語料**。

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

#@title 選擇微調資料集規模 (Dataset Selection)
#@markdown 支援「10萬筆黃金加權平衡庫 (手寫筆記+真題重點加權，強烈推薦)」、「100萬筆全真統一大庫」或「5,082筆純核實核心庫」：
DATASET_MODE = "10萬筆黃金加權平衡庫 (Balanced SFT - 推薦)" #@param ["10萬筆黃金加權平衡庫 (Balanced SFT - 推薦)", "100萬筆全真統一大庫 (1M+ Unified SFT)", "5,082筆純核實核心庫 (Verified SFT)"]

if "加權" in DATASET_MODE or "Balanced" in DATASET_MODE:
    target_file = "law_exam_balanced_weighted_sft.jsonl"
elif "100萬" in DATASET_MODE:
    target_file = "law_exam_master_sft_unified.jsonl"
else:
    target_file = "law_exam_verified_sft.jsonl"

# 嚴格路徑比對：僅搜尋所選 target_file，停用跨資料集隱式降級回退
drive_candidates = [
    f'/content/drive/MyDrive/LawExamData/training_data/{target_file}',
    f'/content/drive/我的雲端硬碟/LawExamData/training_data/{target_file}',
    f'/content/LawExamData/training_data/{target_file}',
    f'./training_data/{target_file}',
    f'./{target_file}'
]

DATA_PATH = None
for p_c in drive_candidates:
    if os.path.exists(p_c):
        DATA_PATH = p_c
        break

if not DATA_PATH:
    raise FileNotFoundError(
        f"❌ 找不到所選微調資料集：{target_file}！\n"
        f"目前選擇模式為：{DATASET_MODE}。\n"
        f"為確保模型訓練純度與可重現性，系統已停用隱式資料集降級回退。\n"
        f"請確認該檔案已位於 Google Drive 之 'LawExamData/training_data/{target_file}'。"
    )

print('✅ 訓練資料集已就緒：', DATA_PATH)

In [ ]:
# 2. 安裝極速微調套件 (包含 unsloth_zoo 與最新依賴)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets triton

import importlib
importlib.invalidate_caches()
print('✅ Unsloth 與 unsloth_zoo 安裝完成！若初次安裝建議重啟工作階段 (Runtime -> Restart session)')


In [ ]:
# 3. 載入基底開源模型或既有 LoRA 權重 (跨 Session 增量微調 Continual SFT / Warm Start)
# 說明：載入既有 LoRA 僅繼承模型適配層權重 (Weights Only)，並以全新優化器展開第二階段增量訓練。
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported
import torch, os, glob, zipfile
from pathlib import Path

max_seq_length = 2048
load_in_4bit = True

#@title 接續先前已訓練 LoRA 權重配置 (可選)
#@markdown 若欲在先前微調過的成果上繼續增量訓練，請填寫 LoRA 目錄或 Zip 檔案路徑：
EXISTING_LORA_PATH = "" #@param {type:"string"}

if EXISTING_LORA_PATH.strip() and os.path.exists(EXISTING_LORA_PATH.strip()):
    lora_target = EXISTING_LORA_PATH.strip()
    if lora_target.endswith(".zip"):
        extract_dir = "/content/previous_lora"
        with zipfile.ZipFile(lora_target, "r") as z:
            z.extractall(extract_dir)
        lora_target = extract_dir
    print(f'🔄 正在載入既有 LoRA 權重以支援【增量持續微調 (Continual Fine-Tuning)】：{lora_target}')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = lora_target,
        max_seq_length = max_seq_length,
        dtype = None,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_training(model)
    print('✅ 既有 LoRA 權重載入完成，已重啟可訓練梯度，準備接續增量微調！')
else:
    print('🚀 載入官方開源基底模型：unsloth/Qwen2.5-7B-Instruct-bnb-4bit')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
        max_seq_length = max_seq_length,
        dtype = None,
        load_in_4bit = load_in_4bit,
    )
    # 配置高效能 LoRA 轉接器
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
    print('✅ Qwen2.5-7B-Instruct 模型與 LoRA 介面全新初始化成功！')


In [ ]:
# 4. 嚴格 5 欄位特徵綱要載入、防止重複過採樣與 Prompt 格式化
from datasets import load_dataset, Features, Value
from trl import SFTTrainer
from transformers import TrainingArguments
import random

# 嚴格定義 5 大標準欄位，保證 100% 綱要相容 (杜絕 CastError)
sft_features = Features({
    "id": Value("string"),
    "source_type": Value("string"),
    "instruction": Value("string"),
    "input": Value("string"),
    "output": Value("string"),
})

# 嚴格模式載入：不進行靜默降級，確保所有樣本均符合 5 欄位規範
dataset = load_dataset('json', data_files=DATA_PATH, split='train', features=sft_features)
print(f'✅ 成功以嚴格綱要載入原始樣本：{len(dataset):,} 筆')

#@title 來源權重動態平衡防護 (防止已加權檔案遭遇二次過採樣)
# 判斷是否為已預先加權之平衡大庫
is_already_weighted = any(k in str(DATA_PATH).lower() for k in ["weighted", "balanced"])

if is_already_weighted:
    print('ℹ️ 偵測到已使用平衡加權庫 (law_exam_balanced_weighted_sft.jsonl)！')
    print('   已自動跳過第二輪重複過採樣，避免資料集膨脹至 650 萬筆導致 RAM OOM！')
else:
    # 僅針對未預先加權之原始庫 (如 verified_sft 或 unified) 執行在線動態加權
    ENABLE_DYNAMIC_WEIGHTING = True #@param {type:"boolean"}
    NOTE_WEIGHT = 500 #@param {type:"integer"} 手寫筆記放大倍率
    EXAM_WEIGHT = 50  #@param {type:"integer"} 司律真題放大倍率
    COURT_WEIGHT = 15 #@param {type:"integer"} 憲法與裁判放大倍率
    
    if ENABLE_DYNAMIC_WEIGHTING and "source_type" in dataset.column_names:
        print('⚖️ 正在對原始未加權語料套用過採樣 (Weighted Oversampling)...')
        sts = dataset["source_type"]
        idx_pool = []
        n_cnt, e_cnt, c_cnt, s_cnt = 0, 0, 0, 0
        for i, s in enumerate(sts):
            st = str(s).lower()
            if "handwritten" in st or "user_note" in st:
                idx_pool.extend([i] * NOTE_WEIGHT)
                n_cnt += NOTE_WEIGHT
            elif "exam" in st or "coursework" in st or "mcq" in st:
                idx_pool.extend([i] * EXAM_WEIGHT)
                e_cnt += EXAM_WEIGHT
            elif "constitutional" in st or "judgment" in st or "rule" in st or "theories" in st:
                idx_pool.extend([i] * COURT_WEIGHT)
                c_cnt += COURT_WEIGHT
            else:
                idx_pool.append(i)
                s_cnt += 1
        random.seed(3407)
        random.shuffle(idx_pool)
        dataset = dataset.select(idx_pool)
        total = len(dataset)
        print(f'✅ 動態加權完成！加權後訓練總樣本數：{total:,} 筆')
        print(f'   - 📝 手寫筆記曝光：{n_cnt:,} 次 ({n_cnt/total*100:.2f}%)')
        print(f'   - 🎓 司律真題/講義：{e_cnt:,} 次 ({e_cnt/total*100:.2f}%)')
        print(f'   - 🏛️ 憲法/裁判規準：{c_cnt:,} 次 ({c_cnt/total*100:.2f}%)')

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, inp, output in zip(instructions, inputs, outputs):
        text = (
            f"<|im_start|>system\n你是一位具備多年命題閱卷經驗之台灣司法官律師考試閱卷委員與法學權威。<|im_end|>\n"
            f"<|im_start|>user\n{instruction}\n{inp}<|im_end|>\n"
            f"<|im_start|>assistant\n{output}<|im_end|>"
        )
        texts.append(text)
    return { "text": texts }

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f'🎉 最終微調訓練集已格式化完成，總計：{len(dataset):,} 筆！')


In [ ]:
# 5. 啟動 SFT 訓練引擎 (支援同環境 Checkpoint 完整斷點續訓)
# 說明：RESUME_FROM_CHECKPOINT 會自動恢復模型權重、AdamW 優化器狀態、學習率排程及步數，實現零損失接續。
from unsloth import is_bfloat16_supported
import os, glob

#@title 訓練步數與接續訓練設定
#@markdown 目標總訓練步數 (若接續訓練，設為大於上次步數的值，例如上次 200 步，這次設 500 或 1000 步)：
max_training_steps = 500 #@param {type:"integer"}
#@markdown 是否自動從上次未跑完或已存檔之 Checkpoint 接續訓練：
RESUME_FROM_CHECKPOINT = True #@param {type:"boolean"}

output_dir = "/content/outputs"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = max_training_steps,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        save_steps = 100, # 每 100 步自動存檔一次 Checkpoint
        save_total_limit = 3, # 最多保留 3 個最新 Checkpoint
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = output_dir,
    ),
)

# 智慧辨識可接續之 Checkpoint
checkpoint_to_resume = None
if RESUME_FROM_CHECKPOINT and os.path.exists(output_dir):
    ckpts = sorted(
        glob.glob(f"{output_dir}/checkpoint-*"),
        key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0
    )
    if ckpts:
        checkpoint_to_resume = ckpts[-1]
        print(f'🔄 偵測到既有 Checkpoint，將自動自 【{checkpoint_to_resume}】 接續訓練至第 {max_training_steps} 步！')
    else:
        print('ℹ️ 未找到既有 Checkpoint，將從 Step 0 啟動全新訓練。')
else:
    print('🚀 啟動全新訓練...')

print('🔥 開始執行訓練流程...')
trainer_stats = trainer.train(resume_from_checkpoint=checkpoint_to_resume)
print('🎉 訓練完成！統計資訊：', trainer_stats)


In [ ]:
# 6. 將微調完成的 LoRA 權重儲存回 Google Drive
SAVE_DIR = '/content/drive/MyDrive/LawExamData/model_weights/qwen_law_evaluator_lora'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print('🎉 司律評分模型 LoRA 權重已永久儲存至 Google Drive：', SAVE_DIR)